In [1]:
# import csv
# from datetime import datetime
# from pathlib import Path

# def save_to_log(fid, experiment_name, log_file_path="evaluation_log.csv"):
#     """
#     Appends FID results and timestamp to a central CSV log.

#     Parameters:
#     - fid: computed FID score (float or str)
#     - experiment_name: name of the experiment (str)
#     - log_file_path: path to the CSV log file (str or Path)
#     """
#     log_file_path = Path(log_file_path)
#     log_exists = log_file_path.exists()

#     timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

#     with open(log_file_path, mode="a", newline="") as f:
#         writer = csv.writer(f)

#         # Write header if the log file is new
#         if not log_exists:
#             writer.writerow(["Timestamp", "Experiment", "FID"])

#         writer.writerow([timestamp, experiment_name, fid])

#     print(f"✅ Logged: {experiment_name} | FID: {fid:.4f} | {timestamp}")

# def find_log_json_path(log_folder: Path) -> Path:
#     json_files = list(log_folder.glob("*.json"))
#     if not json_files:
#         raise FileNotFoundError(f"No .json file found in {log_folder}")
#     if len(json_files) > 1:
#         print(f"⚠️ Multiple .json files found in {log_folder}, using the first one: {json_files[0].name}")
#     return json_files[0]

In [2]:
from pathlib import Path
from datetime import datetime
import json
import os



def find_log_json_path(log_folder: Path) -> Path:
    json_files = list(log_folder.glob("*.json"))
    if not json_files:
        raise FileNotFoundError(f"No .json file found in {log_folder}")
    if len(json_files) > 1:
        print(f"⚠️ Multiple .json files found in {log_folder}, using the first one: {json_files[0].name}")
    return json_files[0]

def save_to_log(fid_value, experiment_name, log_file="global_fid_log.txt"):
    log_entry = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | {experiment_name} | FID: {fid_value:.4f}\n"
    with open(log_file, "a") as f:
        f.write(log_entry)

def run_experiments_from_logs(
    experiment_names,
    GAN,
    generate_images,
    compute_fid,
    HOME_DIR,
    base_log_path,
    base_generation_path_root,
    base_data_path,
    test_dataset_path,
    step: int,
    num_samples=100,
    fix_with_topo=False,
):
    """
    Run multiple experiments by loading their settings from corresponding JSON logs.

    Parameters:
    - experiment_names: list of experiment directory names (folder names in LOG/)
    - GAN, generate_images, compute_fid: callable functions
    - HOME_DIR: string path to restore working directory
    - base_log_path, base_generation_path_root, base_data_path: Path roots
    - num_samples: how many images to generate
    - fix_with_topo: whether to enable topology correction during generation
    """
    for experiment_name in experiment_names:
        print(f"\n▶ Running experiment: {experiment_name}")

        log_folder = base_log_path / experiment_name
        try:
            log_json_path = find_log_json_path(log_folder)
        except FileNotFoundError as e:
            print(f"❌ {e}")
            continue

        with open(log_json_path) as f:
            config = json.load(f)
            

        data_path = Path(config["settings"]["Path"]["data_path"])
        dataset_name = data_path.parts[-2]
        base_generation_path = base_generation_path_root / dataset_name

        weights_path = log_folder / "models"
        generations_base = base_generation_path / experiment_name

        general = config.get("general", {})
        adv_params = config.get("advanced_params", {})
        G_arch = config.get("generator_arch", [])
        D_arch = config.get("discriminator_arch", [])
        
        parser = Argparser(config["settings"], continue_model=True)

        configs = [
            ("general", general),
            ("adv_params", adv_params),
            ("G_arch", G_arch),
            ("D_arch", D_arch),
        ]

        for var_name, cfg in configs:
            if not cfg:
                print(f"⚠️ Warning: config '{var_name}' is empty for experiment '{experiment_name}'")

        current_time = datetime.now().strftime("%Y.%m.%d_%H.%M.%S")
        generations_path = generations_base.with_name(f"{generations_base.stem}_{current_time}")

        gan_kwargs = {
            "general": general,
            "adv_params": adv_params,
            "G_arch": G_arch,
            "D_arch": D_arch,
        }

        # training_dataset_path = data_path
        # step = config.get("settings", {}).get("Basic", {}).get("model_step", 0)

        generate_images(
            GAN,
            weights_path,
            generations_path,
            gan_kwargs=gan_kwargs,
            step=step,
            num_samples=num_samples,
            fix_with_topo=fix_with_topo,
        )

        os.chdir(HOME_DIR)
        fid = calculate_fid_given_paths(paths=([str(generations_path), str(test_dataset_path)]), cuda=True, 
                                        batch_size=50, dims=2048)
        print(f"FID: {fid:.4f}")

        save_to_log(fid, experiment_name)
